# L7.1 · SmolVLA: el mapa

| Notebook | Qué abre |
|---|---|
| **L7.1** | El mapa: qué entra, qué sale, dónde están los 450 M de parámetros |
| L7.2 | La imagen: de la foto a 64 tokens |
| L7.3 | La fila del VLM: palabras, estado y quién puede mirar a quién |
| L7.4 | Adentro del VLM: qué miran las palabras, por qué 16 de 32 capas, qué queda guardado |
| `06` (flujo) | Entrenarlo con nuestros datos |
| L7.5 | El experto de acciones: la brújula de L6 con ojos |
| L7.6 | Las 10 pasadas: un bloque de verdad pasando de ruido a movimiento |
| L7.7 | El entrenamiento por dentro |
| `07` (flujo) | Evaluación contra ACT, en los mismos episodios |

Todo con el modelo real (`lerobot/smolvla_base`, LeRobot 0.4.1) y nuestras
grabaciones del G1. Las piezas salen de `toolkit/policy/smolvla_internals.py`,
cuyos tests verifican que encadenadas dan exactamente lo mismo que el modelo
entero.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import DATASETS_DIR
from toolkit.policy import smolvla_internals as vla
from toolkit.policy.smolvla_g1_base import G1_BASE_DIR, load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"
FPS = 30
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT,
                         delta_timestamps={"action": [i / FPS for i in range(50)]})
names = dataset.meta.features["observation.state"]["names"]
joint_names = names[0] if isinstance(names[0], list) else names  # la conversión las guarda anidadas
policy, preprocess, postprocess = load_g1_policy(G1_BASE_DIR, dataset.meta.stats, device)
model = policy.model

## 1. Por qué un VLA

ACT, nuestra primera política, es un **especialista**: una ResNet entrenada con
fotos de ImageNet más un transformer que aprendió desde cero con 54 episodios.
Todo lo que sabe del mundo es lo que vio en esos episodios. No sabe qué es un
mameluco, no entiende una instrucción, y si cambia la luz o aparece un objeto
nuevo, no tiene de dónde agarrarse.

Mientras tanto existen modelos de visión y lenguaje (VLM) entrenados con
millones de imágenes y textos, que ya saben describir una escena. La idea de un
VLA es aprovechar ese cerebro: **un VLM más una parte que mueve el robot**.

En tus notas hay dos caminos para la parte que mueve. SmolVLA es el **camino
2**, el de π0: el VLM entiende la escena y un **experto de acciones** aparte
genera el bloque con flow matching (la serie L6). Nada de acciones como
palabras.

## 2. Qué entra y qué sale

Agarramos un instante de un episodio en el medio de la tarea. **Entran** tres
cosas: la foto de la cámara de la cabeza, la instrucción en texto y los 26
ángulos actuales del robot. **Sale** un bloque de 50 acciones (los próximos 50
cuadros, 1,7 segundos a 30 por segundo), cada una con los 26 ángulos objetivo.

Abajo, a la derecha, lo que hizo la persona en esos 50 cuadros: lo que el
modelo tendría que aprender a generar.

In [ ]:
episode = 10
frame_index = int(dataset.meta.episodes["dataset_from_index"][episode]) + 120
frame = dataset[frame_index]
print("instrucción:", repr(frame["task"]))
print("estado     :", tuple(frame["observation.state"].shape), "· bloque real:", tuple(frame["action"].shape))

fig = plt.figure(figsize=(16, 4.2))
ax_img = fig.add_subplot(1, 3, 1)
ax_img.imshow(frame[CAMERA].permute(1, 2, 0).numpy())
ax_img.set_title("entra · la cámara de la cabeza")
ax_img.axis("off")
ax_state = fig.add_subplot(1, 3, 2)
ax_state.barh(range(26), frame["observation.state"].numpy(), color="tab:orange")
ax_state.set_yticks(range(26), joint_names, fontsize=6)
ax_state.invert_yaxis()
ax_state.set_title("entra · los 26 ángulos actuales (rad)")
ax_out = fig.add_subplot(1, 3, 3)
chunk = frame["action"].numpy()
for j in range(26):
    ax_out.plot(np.arange(50) / FPS, chunk[:, j], lw=1)
ax_out.set_xlabel("segundos hacia adelante")
ax_out.set_title("sale · bloque de 50 acciones × 26 ángulos")
plt.tight_layout()

## 3. Dónde están los 450 millones de parámetros

Contamos los parámetros de cada parte en el modelo cargado:

In [ ]:
def millions(module):
    return sum(p.numel() for p in module.parameters()) / 1e6


vlm = model.vlm_with_expert.get_vlm_model()
parts = {
    "visión (SigLIP, 12 capas)": millions(vlm.vision_model),
    "conector": millions(vlm.connector),
    "VLM de texto (16 de 32 capas)": millions(vlm.text_model),
    "experto de acciones (16 capas)": millions(model.vlm_with_expert.lm_expert),
    "proyecciones (estado, acciones, tiempo)": sum(millions(m) for m in [model.state_proj, model.action_in_proj,
        model.action_out_proj, model.action_time_mlp_in, model.action_time_mlp_out]),
    "cabeza de palabras del VLM (no se usa)": millions(model.vlm_with_expert.vlm.lm_head),
}
trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad) / 1e6

fig, ax = plt.subplots(figsize=(9, 3))
colors = ["tab:blue", "tab:cyan", "tab:green", "tab:red", "tab:orange", "lightgray"]
ax.barh(list(parts), list(parts.values()), color=colors)
for i, value in enumerate(parts.values()):
    ax.text(value + 3, i, f"{value:.1f} M", va="center")
ax.invert_yaxis()
ax.set_xlabel("millones de parámetros")
ax.set_title(f"total {sum(parts.values()):.0f} M · se entrenan {trainable:.0f} M")
assert abs(sum(parts.values()) - millions(policy)) < 1, "falta contar alguna parte"
plt.tight_layout()
words = vlm.text_model.get_input_embeddings().weight
print(f"de los {parts['VLM de texto (16 de 32 capas)']:.0f} M del VLM de texto, {words.numel() / 1e6:.0f} M son la tabla de "
      f"palabras: {words.shape[0]} palabras × {words.shape[1]} números")

Para mirar:

- **El grueso es el VLM** (visión, conector y texto: unos 300 M) y queda
  **congelado**: al entrenar nadie lo toca. Se entrenan sólo el experto y las
  proyecciones, 100 M.
- **Un cuarto del VLM de texto es la tabla de palabras**: 49.280 palabras ×
  960 números. Es el diccionario: cada palabra posible tiene su vector.
- **47 M se cargan y no se usan nunca**: la cabeza de palabras, la capa que en
  un VLM elige la próxima palabra. SmolVLA no escribe texto, así que no la
  necesita.
- Para comparar, nuestra ACT tiene unos 15 M (L1).

## 4. El viaje de una observación

Todo termina siendo filas de vectores. Hay **dos filas**:

- **La fila del VLM**: los tokens de la imagen, los de la instrucción y uno
  del estado. Pasa **una sola vez** por las 16 capas del VLM, y de cada capa
  queda guardado lo necesario para que otros la consulten (L7.4).
- **La fila del experto**: 50 casilleros, uno por acción del bloque. Arrancan
  como ruido y pasan **10 veces** por las 16 capas del experto, que en cada
  capa consulta lo guardado del VLM. Cada pasada es un pasito de la brújula de
  L6.3.

Así se ven, con los largos reales de nuestro instante:

In [ ]:
batch = preprocess(dict(frame))
obs = vla.prepare_observation(policy, batch)
with torch.no_grad():
    prefix = vla.embed_prefix(model, obs)
sizes = {name: s.stop - s.start for name, s in prefix.segments.items()}
real_words = int(obs.lang_masks.sum())
labels = {"imagen": "imagen\n64", "texto": f"texto: {sizes['texto']}\n({real_words} de palabras, el resto relleno)", "estado": "estado: 1"}

fig, ax = plt.subplots(figsize=(13, 2.6))
start = 0
for (name, size), color in zip(sizes.items(), ["tab:blue", "tab:green", "tab:orange"]):
    ax.barh(1, size, left=start, color=color, edgecolor="white")
    if size > 5:
        ax.text(start + size / 2, 1, labels[name], ha="center", va="center", color="white", fontsize=9)
    else:
        ax.annotate(labels[name], (start + size / 2, 1.45), ha="center", fontsize=9)
    start += size
ax.barh(0, 50, left=0, color="tab:red", edgecolor="white")
ax.text(25, 0, "50 casilleros de acción (ruido → bloque)", ha="center", va="center", color="white", fontsize=9)
ax.set_yticks([1, 0], ["fila del VLM\n1 vez por las 16 capas", "fila del experto\n10 veces por las 16 capas"])
ax.set_xlabel("tokens")
ax.set_xlim(0, start + 5)
plt.tight_layout()
ax.set_ylim(-0.6, 1.9)
print("fila del VLM:", sizes, "· palabras de verdad:", real_words, "· ancho de cada token:", prefix.embs.shape[-1])

## 5. El modelo base con nuestro robot

`smolvla_base` se preentrenó con unos 23.000 episodios de brazos SO-100: brazos
chicos de **6 articulaciones**, con tres cámaras. Nuestro G1 tiene 26. ¿Qué
genera para nuestro instante? Le pedimos un bloque y lo comparamos con lo que
hizo la persona, en cuatro articulaciones del brazo derecho y la mano:

In [ ]:
with torch.no_grad():
    generated = postprocess(policy.predict_action_chunk(batch))[0].cpu().numpy()

watch = [joint_names.index(n) for n in joint_names if n.startswith(("kRightShoulderPitch", "kRightElbow", "kRightWristRoll"))][:3]
watch += [j for j, n in enumerate(joint_names) if "right" in n.lower() and "thumb" in n.lower()][:1]
fig, axes = plt.subplots(1, len(watch), figsize=(16, 3.2))
t = np.arange(50) / FPS
for ax, j in zip(axes, watch):
    ax.plot(t, chunk[:, j], color="black", label="la persona")
    ax.plot(t, generated[:, j], color="tab:red", label="smolvla_base")
    ax.set_title(joint_names[j], fontsize=9)
    ax.set_xlabel("s")
axes[0].legend(fontsize=8)
plt.tight_layout()
error = np.abs(generated - chunk).mean()
print(f"error medio: {error:.2f} rad ({np.degrees(error):.0f}°)")

El modelo base devuelve **líneas planas**: una postura fija, cerca del
promedio de nuestros datos, que no se mueve. Su experto aprendió a mover brazos
de 6 articulaciones: para las otras 20 nunca vio nada, y las primeras 6 de un
SO-100 no significan lo mismo que las del G1 (allá la sexta es una pinza, acá
es una muñeca). Sin datos nuestros, no tiene de dónde sacar el movimiento.

Lo que sí sirve tal cual es todo lo que **entiende**: la visión y el texto
vienen del VLM, que vio millones de imágenes, y quedan congelados. Lo que hay
que enseñarle es a **mover**: eso es entrenar el experto (`06`).

## Lo que nos llevamos

- Un VLA de camino 2 es un VLM que entiende la escena más un experto que
  genera el bloque con flow matching.
- Entran foto, instrucción y 26 ángulos; sale un bloque de 50 × 26.
- De 450 M de parámetros se entrenan 100 M: el experto y las proyecciones. El
  VLM queda congelado, y por eso las L7.2 a L7.4 (que miran el VLM) valen igual
  antes y después de entrenarlo.
- El modelo base no sabe mover un G1: hay que entrenarlo con nuestros datos
  (`06`).

## Referencias

- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025).
- Black et al., [π0](https://arxiv.org/abs/2410.24164) (2024): el camino 2.
- [Ficha de `smolvla_base`](https://huggingface.co/lerobot/smolvla_base) y
  [documentación de LeRobot](https://huggingface.co/docs/lerobot/smolvla).